# VibeShift Flow Matching Training Notebook

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
from tqdm.notebook import tqdm

from training import Trainer, TrainingConfig
from dataloader import create_dataloader, GenreAwareLatentDataset



In [ ]:
SOURCE_DIR = "data/latent_data/latent_classical"
TARGET_DIR = "data/latent_data/latent_synth1"
BATCH_SIZE = 16
NUM_WORKERS = 4
MAX_SAMPLES = None
SHUFFLE = True
DROP_LAST = True
TRAIN_VAL_SPLIT = 0.9  # 90% train, 10% val

# Import required utilities
from torch.utils.data import Subset, DataLoader
from dataloader import default_collate_with_dynamic_padding

# Load full dataset first
full_loader, full_dataset = create_dataloader(
    source_dir=SOURCE_DIR,
    target_dir=TARGET_DIR,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE,
    num_workers=NUM_WORKERS,
    max_samples=MAX_SAMPLES,
    drop_last=DROP_LAST,
    pin_memory=True,
)

# Split into train/val
dataset_size = len(full_dataset)
train_size = int(dataset_size * TRAIN_VAL_SPLIT)
val_size = dataset_size - train_size

print(f"Dataset: {dataset_size} samples")
print(f"  Train: {train_size} samples (90%)")
print(f"  Val:   {val_size} samples (10%)")

# Create train/val dataloaders from the full dataset
train_indices = list(range(train_size))
val_indices = list(range(train_size, dataset_size))

train_subset = Subset(full_dataset, train_indices)
val_subset = Subset(full_dataset, val_indices)

# Use the same collate function for dynamic padding
train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE,
    num_workers=NUM_WORKERS,
    drop_last=DROP_LAST,
    pin_memory=True,
    collate_fn=default_collate_with_dynamic_padding,
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Don't shuffle validation
    num_workers=NUM_WORKERS,
    drop_last=False,
    pin_memory=True,
    collate_fn=default_collate_with_dynamic_padding,
)

print(f"Batch size: {BATCH_SIZE}")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"Val batches per epoch: {len(val_loader)}")

info = full_dataset.get_info()
print(f"Embedding dim: {info['embedding_dim']}")

# Validate data with sample batch
try:
    batch = next(iter(train_loader))
    print(f"Batch elements: {len(batch)}")
    
    x0_sample = batch[0]
    x1_sample = batch[1]
    genre_ids = None
    mask = None
    
    # Detect third element by dtype
    if len(batch) > 2:
        if batch[2].dtype in [torch.long, torch.int64]:
            genre_ids = batch[2]
            mask = batch[3] if len(batch) > 3 else None
        elif batch[2].dtype == torch.float32:
            mask = batch[2]
    
    print(f"[OK] Sample batch loaded")
    print(f"  x0 shape: {x0_sample.shape}, range: [{x0_sample.min():.4f}, {x0_sample.max():.4f}]")
    print(f"  x1 shape: {x1_sample.shape}, range: [{x1_sample.min():.4f}, {x1_sample.max():.4f}]")
    if genre_ids is not None:
        print(f"  genre_ids: {genre_ids.shape}, unique: {genre_ids.unique().tolist()}")
    if mask is not None:
        print(f"  mask shape: {mask.shape}, valid ratio: {mask.sum() / mask.numel():.2%}")
except Exception as e:
    print(f"[WARNING] Could not load sample batch: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
try:
    from models.dit import DiT
    from models.flow import FlowMatching
    
    DIT_CONFIG = {
        "input_dim": 1024,
        "embed_dim": 512,
        "num_blocks": 8,
        "num_heads": 8,
        "num_genres": 3,
    }
    
    # Validate embedding dimension matches dataloader
    if info['embedding_dim'] != DIT_CONFIG['input_dim']:
        raise ValueError(f"Dimension mismatch! Dataset: {info['embedding_dim']}, Model: {DIT_CONFIG['input_dim']}")
    
    dit = DiT(**DIT_CONFIG)
    model = FlowMatching(dit)
    
    # Move to device FIRST
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(DEVICE)
    
    print(f"[OK] Model loaded and moved to {DEVICE}")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"[OK] Dimension validation passed")
except ImportError as e:
    print(f"[WARNING] Could not import models: {e}")
    print("Using dummy model for demonstration")
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    model = nn.Linear(512, 512).to(DEVICE)
except ValueError as e:
    print(f"[ERROR] {e}")
    raise


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EXPERIMENT_NAME = "vibeshift_flow_matching"

optimizer = torch.optim.AdamW(model.parameters())

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=100,
    eta_min=1e-6,
)

def create_loss_fn(model):
    """
    Create a loss function that wraps the model's loss computation.
    Detects capability once to avoid per-batch exception overhead.
    """
    has_compute_loss = hasattr(model, "compute_loss")

    def loss_fn(x0, x1, genre_ids, mask=None):
        if has_compute_loss:
            # FlowMatching path: supports mask argument
            return model.compute_loss(x0, x1, genre_ids, mask=mask)
        else:
            # Generic nn.Module path: no mask support
            return model(x0, x1, genre_ids)
    return loss_fn

loss_fn = create_loss_fn(model)

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    device=DEVICE,
    checkpoint_dir="checkpoints",
    name=EXPERIMENT_NAME,
    use_amp=True if DEVICE == "cuda" else False,
)

print(f"[OK] Trainer initialized")
print(f"Device: {DEVICE}")
print(f"AMP: {trainer.use_amp}")
print(f"Checkpoint dir: {trainer.checkpoint_dir}")



In [ ]:
LOAD_CHECKPOINT = False
CHECKPOINT_PATH = None

if LOAD_CHECKPOINT and CHECKPOINT_PATH is not None:
    checkpoint_path = Path(CHECKPOINT_PATH)
    
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"[ERROR] Checkpoint not found at {checkpoint_path}")
    
    try:
        start_epoch = trainer.load_checkpoint(str(checkpoint_path), load_optimizer=True)
        print(f"[OK] Training resumed from epoch {start_epoch}")
    except KeyError as e:
        print(f"[ERROR] Checkpoint corrupted or incompatible: {e}")
        raise
    except Exception as e:
        print(f"[ERROR] Failed to load checkpoint: {e}")
        raise
else:
    print("Starting fresh training (no checkpoint loaded)")
    start_epoch = 0


In [ ]:
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 20  # Stop if val loss doesn't improve for 20 epochs

print("Starting training with validation...")
with tqdm(total=NUM_EPOCHS, desc="Training Progress", unit="epoch") as pbar:
    results = trainer.train(
        train_dataloader=train_loader,
        num_epochs=NUM_EPOCHS,
        loss_fn=loss_fn,
        val_dataloader=val_loader,  # Now using validation dataloader
        scheduler=scheduler,
        gradient_clip=1.0,
        accumulation_steps=4,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        log_interval=5,
        monitor_gradients=True,
        save_interval=10,  # Saves periodic checkpoints every 10 epochs + best checkpoint
    )
    pbar.update(NUM_EPOCHS)

# Validate results dict
assert isinstance(results, dict), "[ERROR] Training did not return results dict"
assert 'epoch_losses' in results, "[ERROR] Missing epoch_losses in results"

num_epochs_trained = len(results['epoch_losses'])

# Display training summary with progress details
print(f"\n[OK] Training completed!")
print(f"  Epochs run: {num_epochs_trained}/{NUM_EPOCHS}")
print(f"  Train loss: {results['epoch_losses'][-1]:.6f}")
if results.get('val_losses') and len(results['val_losses']) > 0:
    print(f"  Val loss: {results['val_losses'][-1]:.6f}")
print(f"  Best loss: {results['best_loss']:.6f} (epoch {results['best_epoch']+1})")
print(f"  Loss reduction: {results['epoch_losses'][0] - results['epoch_losses'][-1]:.6f}")
print(f"  Gradient norms tracked: {len(results['gradient_norms'])}")
print(f"  Checkpoints saved to: {trainer.checkpoint_dir}")



In [ ]:
import matplotlib.pyplot as plt

if not isinstance(results, dict) or 'epoch_losses' not in results:
    print("[WARNING] Cannot visualize: results dict not available or incomplete")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    epochs = range(1, len(results['epoch_losses']) + 1)
    axes[0, 0].plot(epochs, results['epoch_losses'], linewidth=2)
    axes[0, 0].set_title('Training Loss per Epoch')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].grid(True, alpha=0.3)
    
    if results.get('learning_rates') and len(results['learning_rates']) > 0:
        axes[0, 1].plot(epochs[:len(results['learning_rates'])], results['learning_rates'], linewidth=2, color='green')
        axes[0, 1].set_title('Learning Rate Schedule')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Learning Rate')
        axes[0, 1].grid(True, alpha=0.3)
    else:
        axes[0, 1].text(0.5, 0.5, 'No learning rate data', ha='center', va='center')
        axes[0, 1].set_title('Learning Rate Schedule')
        axes[0, 1].axis('off')
    
    if results.get('gradient_norms') and len(results['gradient_norms']) > 0:
        axes[1, 0].plot(results['gradient_norms'], linewidth=1, alpha=0.7, color='orange')
        axes[1, 0].set_title('Gradient Norms')
        axes[1, 0].set_xlabel('Step')
        axes[1, 0].set_ylabel('Norm')
        axes[1, 0].grid(True, alpha=0.3)
    else:
        axes[1, 0].text(0.5, 0.5, 'No gradient data', ha='center', va='center')
        axes[1, 0].set_title('Gradient Norms')
        axes[1, 0].axis('off')
    
    loss_improvement = results['epoch_losses'][0] - results['epoch_losses'][-1]
    summary_text = f"Training Summary:\n\n"
    summary_text += f"Best Loss: {results['best_loss']:.6f}\n"
    summary_text += f"Best Epoch: {results['best_epoch']}\n"
    summary_text += f"Total Epochs: {len(results['epoch_losses'])}\n"
    summary_text += f"Final Loss: {results['epoch_losses'][-1]:.6f}\n"
    summary_text += f"Loss Improvement: {loss_improvement:.6f}"
    
    axes[1, 1].text(0.1, 0.9, summary_text,
                    transform=axes[1, 1].transAxes,
                    fontsize=11,
                    verticalalignment='top',
                    fontfamily='monospace',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n[OK] Visualization complete")
    print(f"Best Loss: {results['best_loss']:.6f} (Epoch {results['best_epoch']})")
    print(f"Final Loss: {results['epoch_losses'][-1]:.6f}")


In [ ]:
# Save final model (best checkpoint)
best_checkpoints = sorted(trainer.checkpoint_dir.glob("best*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)

if best_checkpoints:
    best_checkpoint = best_checkpoints[0]
    
    # Load best weights into current model
    trainer.load_checkpoint(str(best_checkpoint), load_optimizer=False)
    
    final_path = trainer.checkpoint_dir / "final_model.pt"
    torch.save(model.state_dict(), final_path)
    print(f"[OK] Final (best) model saved to {final_path}")
    print(f"[OK] Based on checkpoint: {best_checkpoint.name}")
else:
    # Fallback: save current model if no best checkpoint found
    print(f"[WARNING] No best checkpoint found, saving current model weights")
    final_path = trainer.checkpoint_dir / "final_model.pt"
    torch.save(model.state_dict(), final_path)
    print(f"[OK] Model saved to {final_path}")

print(f"\n[OK] All checkpoints saved in: {trainer.checkpoint_dir}")
all_checkpoints = list(trainer.checkpoint_dir.glob("*.pt"))
print(f"Total checkpoint files: {len(all_checkpoints)}")

